# ICOM5015: Adversarial AI in 4-Player Partnership Domino
**Team Members:** Keniel (Rules), Ernesto (State Management), Noel (Scoring), Member 4 (Interface, Integration & Evaluation)

## 1. AI Approach & Library Justification
- **Game Selected:** 4-Player Partnership Domino (cross-table teams: Team 1 = Players 0 & 2; Team 2 = Players 1 & 3; target = 100 points).
- **AI Techniques Used (Syllabus #3 & #4):** **Averaging Over Clairvoyance** combined with **UCB1 Monte Carlo Simulation**. Pure Minimax is intractable due to $\binom{21}{7}\binom{14}{7} \approx 3.99 \times 10^8$ hidden deal states at Turn 1. Sampling consistent worlds constrained by observed player passes and running tactical Monte Carlo rollouts enables principled decision-making under partial observability.
- **Library Used:** **PettingZoo (`AECEnv`)** — natively models turn-based, multi-agent partial observability (`observe(agent)`) and team rewards without exposing hidden opponent hands.

In [ ]:
%pip install matplotlib pettingzoo gymnasium

In [ ]:
from ai_agents import ClairvoyanceMCTSAgent, HeuristicAgent
from game_interface import DominoGameController

demo_agents = [
    ClairvoyanceMCTSAgent(num_worlds=20, rollouts_per_world=8, seed=42, name="ClairvoyanceAI-P0"),
    HeuristicAgent("Heuristic-P1"),
    ClairvoyanceMCTSAgent(num_worlds=20, rollouts_per_world=8, seed=43, name="ClairvoyanceAI-P2"),
    HeuristicAgent("Heuristic-P3"),
]

demo_controller = DominoGameController(
    agents=demo_agents,
    target_score=1,
    reveal_all_hands=True,
    turn_delay_sec=0.0,
)
demo_result = demo_controller.play_match(seed=42)

In [ ]:
from ai_agents import run_full_tournament

# Evaluate across 100 full 100-point matches per scenario (700 matches total)
tournament_results = run_full_tournament(num_matches=100, seed=42)

In [ ]:
import matplotlib.pyplot as plt

labels = [
    "AI+AI vs\nRand+Rand",
    "AI+AI vs\nHeur+Heur",
    "AI+Rand vs\nRand+Rand",
    "AI+Rand vs\nHeur+Heur",
    "AI+Heur vs\nHeur+Heur",
    "AI(NoInfer)+AI\nvs Heur+Heur",
    "AI+AI vs\nAI+AI (Self)",
]
win_rates = [r["t1_win_rate"] for r in tournament_results]
score_diffs = [r["mean_score_diff"] for r in tournament_results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Team 1 Win Rate (%)
bars1 = axes[0].bar(labels, win_rates, color=["#2b5c8f", "#2b5c8f", "#5c8fb8", "#d95f02", "#7570b3", "#66a61e", "#7f7f7f"])
axes[0].axhline(50, color="red", linestyle="--", linewidth=1, label="50% Parity Baseline")
axes[0].set_ylim(0, 105)
axes[0].set_ylabel("Team 1 Match Win Rate (%)")
axes[0].set_title("Partnership Win Rate Across Opponents & Partner Skill")
axes[0].tick_params(axis="x", rotation=25)
axes[0].legend()
for bar, val in zip(bars1, win_rates):
    axes[0].text(bar.get_x() + bar.get_width() / 2, val + 1.5, f"{val:.1f}%", ha="center", fontsize=9)

# Plot 2: Mean Score Differential
colors2 = ["#2b5c8f" if d >= 0 else "#d95f02" for d in score_diffs]
bars2 = axes[1].bar(labels, score_diffs, color=colors2)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_ylabel("Mean Final Score Differential (Team 1 - Team 2)")
axes[1].set_title("Average Match Margin (100-Point Games)")
axes[1].tick_params(axis="x", rotation=25)
for bar, val in zip(bars2, score_diffs):
    offset = 2.0 if val >= 0 else -5.5
    axes[1].text(bar.get_x() + bar.get_width() / 2, val + offset, f"{val:+.1f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("domino_tournament_evaluation.png", dpi=300)
plt.show()